# Guac training run on Google Colab

This notebook runs the Guac arithmetic pipeline end to end: **pretraining → arithmetic mid-training → evaluation**. It drives the repo's own `make` targets, so the outputs match a local run.

The defaults use `guac-106m-arith`, a small model with a character tokenizer. That tokenizer is built from code, so no tokenizer training or large download is needed, and the run fits on a free Colab T4 GPU.

**Before you start**

1. Set the runtime: **Runtime → Change runtime type → T4 GPU**.
2. Edit the settings in the next cell, then run all cells top to bottom (**Runtime → Run all**).
3. Keep `USE_DRIVE = True` unless you only want a throwaway run. Colab's local disk is erased when the runtime disconnects, and Drive keeps the datasets, tokenizer, checkpoints and models.

In [ ]:
# --- Settings: the only cell you should need to edit ---
REPO_URL = "https://github.com/stanier/guac.git"
BRANCH = "phase-1-mamba2"            # branch to train from
REPO_DIR = "/content/guac"           # where the repo is cloned on the Colab disk

USE_DRIVE = True                     # keep scratch/ (datasets, tokenizers, checkpoints, models) on Google Drive
DRIVE_SCRATCH = "/content/drive/MyDrive/guac/scratch"

CONFIG = "guac-106m-arith"           # preset name in model_configs.yaml
PRETRAIN_DATASETS = "wikitext-103"   # built-in name; others come from dataset_configs in site_config.yml
PRETRAIN_MAX_STEPS = 2000            # optimizer steps for pretraining
PRETRAIN_EXTRA_ARGS = ""             # extra pretrain.py flags, e.g. "--resume-from-checkpoint <dir>"
MIDTRAIN_MAX_STEPS = 1000            # optimizer steps for arithmetic mid-training
EVAL_PER_CATEGORY = 50               # arithmetic eval problems per category

In [ ]:
import os
import shutil
import subprocess


def run(cmd, cwd=None):
    """Run a shell command in the repo, stream its output, and stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, cwd=cwd or REPO_DIR, executable="/bin/bash")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")

In [ ]:
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError("No GPU attached. Use Runtime → Change runtime type → T4 GPU, then run again.")
print("GPU:", result.stdout.strip())

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 1. Get the code

The notebook clones the branch named in `BRANCH`. If the repository is private, put a token in the clone URL using Colab Secrets rather than pasting it into a cell.

In [ ]:
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
else:
    run(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}", cwd="/content")
run("git log --oneline -3")

## 2. Install dependencies

This installs the project in editable mode, which pulls in the pinned `transformers==4.46.3` and `trl` versions from `pyproject.toml`. Colab's preinstalled PyTorch is kept. Every stage runs as a separate process, so the notebook kernel does not need a restart.

In [ ]:
run("pip install -q -e .")
run('python -c "import transformers, torch; print(transformers.__version__, torch.__version__, torch.cuda.is_available())"')

## 3. Configure the working directories

The pipeline reads and writes `scratch/` inside the repo. With `USE_DRIVE = True`, that folder becomes a symlink to Google Drive, so the outputs survive a disconnect.

In [ ]:
os.chdir(REPO_DIR)  # the path manager and the Makefile resolve relative paths from the repo root

if not os.path.exists("site_config.yml"):
    shutil.copy("site_config.example.yml", "site_config.yml")

if USE_DRIVE:
    os.makedirs(DRIVE_SCRATCH, exist_ok=True)
    if os.path.lexists("scratch") and not os.path.islink("scratch"):
        raise RuntimeError("scratch/ already exists in the clone. Delete it, or set USE_DRIVE = False.")
    if not os.path.islink("scratch"):
        os.symlink(DRIVE_SCRATCH, "scratch")
os.makedirs("scratch", exist_ok=True)
print("scratch ->", os.path.realpath("scratch"))

## 4. Build the tokenizer

`guac-106m-arith` uses a character-level tokenizer, which is built from code in a few seconds. The larger presets use a 60k SentencePiece tokenizer instead, which must be trained first. See the notes at the end.

In [ ]:
run("python -m scripts.training.tokenizer.build_char_tokenizer")

## 5. Pretraining

Trains the foundation language model on `PRETRAIN_DATASETS`. The first run tokenizes the dataset and caches the result under `scratch/`. Later runs reuse the cache. Progress is printed as training runs.

In [ ]:
run(f'make pretrain CONFIG={CONFIG} DATASETS={PRETRAIN_DATASETS} MAX_STEPS={PRETRAIN_MAX_STEPS} PRETRAIN_ARGS="{PRETRAIN_EXTRA_ARGS}"')

## 6. Arithmetic mid-training

Loads the pretrained checkpoint and trains it on generated arithmetic problems. The problems are generated locally, so this stage needs no download.

In [ ]:
run(f'make midtrain CONFIG={CONFIG} TASK=arithmetic MIDTRAIN_ARGS="--max-steps {MIDTRAIN_MAX_STEPS}"')

## 7. Evaluate

Measures exact-match accuracy on held-out arithmetic problems, with `EVAL_PER_CATEGORY` problems per category.

In [ ]:
run(f"make eval-arithmetic CONFIG={CONFIG} TASK=arithmetic EVAL_N={EVAL_PER_CATEGORY}")

## 8. Where the outputs are

Pretrained, mid-trained and checkpoint folders are listed below. With `USE_DRIVE = True` they are also in `MyDrive/guac/scratch/` in your Drive.

In [ ]:
run("find scratch/models -mindepth 2 -maxdepth 3 -type d | sort")

## Notes

- **Resuming after a disconnect.** Run the setup cells again (they skip work that is already done), then set `PRETRAIN_EXTRA_ARGS` to `--resume-from-checkpoint <path>` and rerun the pretraining cell. Use a `checkpoint-N` folder from the pretraining output directory. Mid-training resumes the same way through `MIDTRAIN_ARGS`.
- **Other presets.** Only `guac-106m-arith` has been run through this notebook. Larger presets need the 60k SentencePiece tokenizer first (`make train-tokenizer`, which downloads data and takes a while). They also need more GPU memory than a T4 provides, so check the preset's batch size in `model_configs.yaml`, and use an A100 or L4 runtime.
- **Other datasets.** `fineweb`, `fineweb-edu` and similar names are defined under `dataset_configs` in `site_config.yml`. The example config in this notebook doesn't include them, so add that section to `site_config.yml` before using those names.
- **Post-training is not included.** The `make posttrain` stage has not been checked in this notebook.